In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv("C:\Auspify Technologies ML intern\Data\Dataset.csv")
df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [3]:
print("Shape", df.shape)
print("-*" * 50)
print("Columns:", df.columns)
print("-*" * 50)
df.info()
print("-*" * 50)
print("Duplicates :", df.duplicated().sum())

Shape (8790, 10)
-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*
Columns: Index(['show_id', 'type', 'title', 'director', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in'],
      dtype='object')
-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8790 non-null   object
 1   type          8790 non-null   object
 2   title         8790 non-null   object
 3   director      8790 non-null   object
 4   country       8790 non-null   object
 5   date_added    8790 non-null   object
 6   release_year  8790 non-null   int64 
 7   rating        8790 non-null   object
 8   duration      8790 non-null   object
 9   listed_in     8790 non-null   object
dtypes: 

### Prepare content-related features

In [4]:
# Select the features we want to use
features = ["director", "country", "rating", "listed_in"]

# Create a copy containing only these features
df_model = df[["title"] + features].copy()


# Handle missing values
for feature in features:
    df_model[feature] = df_model[feature].fillna("Unknown")


# Clean the feature values

# Convert everything to string
for feature in features:
    df_model[feature] = df_model[feature].astype(str)

# Replace commas with spaces
for feature in features:
    df_model[feature] = df_model[feature].str.replace(",", " ", regex=False)

# Replace spaces inside multi-word values with underscores
for feature in features:
    df_model[feature] = df_model[feature].str.replace(" ", "_", regex=False)


# Step 5: Combine all features into one text column
df_model["combined_features"] = (
    df_model["director"] + " " +
    df_model["country"] + " " +
    df_model["rating"] + " " +
    df_model["listed_in"]
)

# Check the result
print(df_model[["title", "combined_features"]].head())

                              title  \
0              Dick Johnson Is Dead   
1                         Ganglands   
2                     Midnight Mass   
3  Confessions of an Invisible Girl   
4                           Sankofa   

                                   combined_features  
0  Kirsten_Johnson United_States PG-13 Documentaries  
1  Julien_Leclercq France TV-MA Crime_TV_Shows__I...  
2  Mike_Flanagan United_States TV-MA TV_Dramas__T...  
3  Bruno_Garotti Brazil TV-PG Children_&_Family_M...  
4  Haile_Gerima United_States TV-MA Dramas__Indep...  


### Convert text data into machine-readable format

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Convert text into TF-IDF vectors

tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(
    df_model["combined_features"]
)

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

TF-IDF matrix shape: (8790, 5400)


###  Calculate content similarity scores

In [6]:

from sklearn.metrics.pairwise import cosine_similarity


# 2. Calculate cosine similarity

similarity_matrix = cosine_similarity(tfidf_matrix)

print(f"Similarity matrix shape: {similarity_matrix.shape}")


# 3. Create Title → Index Mapping

indices = pd.Series(
    df_model.index,
    index=df_model["title"]
).drop_duplicates()


Similarity matrix shape: (8790, 8790)


### Generate recommendations for selected titles

In [7]:
# 4. Recommendation Function

def recommend(title, n=10):

    # Check whether the title exists
    if title not in indices:
        return f"'{title}' not found."

    # Get the index of the selected title
    idx = indices[title]

    # Get similarity scores for this title
    sim_scores = similarity_matrix[idx]

    # Sort indexes by similarity score
    sim_indices = sim_scores.argsort()[::-1]

    # Remove the selected title itself
    sim_indices = sim_indices[1:n + 1]

    # Return recommended titles
    recommendations = df_model.iloc[
        sim_indices
    ]["title"]

    return recommendations.tolist()

### Evaluate recommendation quality

In [8]:
# 5. Test the Recommendation System

recommendations = recommend(
    "Naruto Shippuden the Movie: Blood Prison",
    n=10
)

print("\nRecommended titles:")

for i, movie in enumerate(recommendations, start=1):
    print(f"{i}. {movie}")


Recommended titles:
1. Naruto Shippuden : Blood Prison
2. Naruto Shippuden: The Movie: The Lost Tower
3. Naruto Shippûden the Movie: The Will of Fire
4. InuYasha the Movie 2: The Castle Beyond the Looking Glass
5. InuYasha the Movie 3: Swords of an Honorable Ruler
6. InuYasha: The Movie 2: The Castle Beyond the Looking Glass
7. Inuyasha the Movie - La spada del dominatore del mondo
8. The Seven Deadly Sins the Movie: Prisoners of the Sky
9. GODZILLA City on the Edge of Battle
10. Ghost Whispers
